# Phase 3 — Forecasting Baselines & Evaluation Protocol

> **Protocol locked before model comparison**
>
> - **Forecast origin:** end of July; assume July's value is available at the end of July.
> - **Horizon:** the next 12 calendar months, **August through July**.
> - **Training boundary:** each forecasting function receives only the target series through that fold's `train_end`.
> - **Scoring mask:** exclude target months from **March 2020 through June 2021**, inclusive.
> - **Selection:** compare candidate drift estimators on the four validation origins only. Keep the two test origins untouched until the validation winner is selected.
> - **Success bar:** pooled RelMAE ≤ **0.90** versus seasonal naïve and beat seasonal naïve in at least **4 of 6** folds. Also report fold-level results because six folds are noisy.
> - **December:** report absolute percentage error; one December per fold is not enough to infer a weekend-placement effect.
>
> **Split diagram**
>
> `Monthly history ──► train through Jul(origin) ──► Aug(origin+1) ... Jul(origin+2)`
>
> `Validation origins: 2016, 2017, 2018, 2019  |  Test origins: 2022, 2023`
>
> The data release lag is a limitation: this protocol assumes July sales are known at July month-end. If operational forecasts are issued before the Census release, shift the origin back to the latest actually available month.


In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.analysis.eda_helper import load_apparel_data
from src.evaluation import (
    VALIDATION_ORIGINS, TEST_ORIGINS, ALL_ORIGINS,
    get_forecast_folds, apply_scoring_mask,
    calculate_mae, calculate_rmse, calculate_bias, calculate_rel_mae,
    seasonal_naive_forecast, make_drift_forecaster, evaluate,
)

df = load_apparel_data().asfreq("MS")
y = df["apparel_sales_nsa"].dropna().astype(float)

print(f"Dataset: {len(y):,} monthly observations, {y.index.min():%Y-%m} to {y.index.max():%Y-%m}")
for fold in get_forecast_folds(ALL_ORIGINS, data_end=y.index.max()):
    months = fold["forecast_months"]
    print(
        f"Origin {fold['origin']:%Y-%m} | train through {fold['train_end']:%Y-%m} | "
        f"forecast {months[0]:%Y-%m} to {months[-1]:%Y-%m}"
    )


Dataset: 415 monthly observations, 1992-01 to 2026-07
Origin 2016-07 | train through 2016-07 | forecast 2016-08 to 2017-07
Origin 2017-07 | train through 2017-07 | forecast 2017-08 to 2018-07
Origin 2018-07 | train through 2018-07 | forecast 2018-08 to 2019-07
Origin 2019-07 | train through 2019-07 | forecast 2019-08 to 2020-07
Origin 2022-07 | train through 2022-07 | forecast 2022-08 to 2023-07
Origin 2023-07 | train through 2023-07 | forecast 2023-08 to 2024-07


In [2]:
# Baseline comparison. Every forecast function receives only `train` and forecast dates.
candidate_forecasters = {
    "Seasonal Naive": seasonal_naive_forecast,
    "Drift: masked mean YoY": make_drift_forecaster("mean_all"),
    "Drift: masked median YoY": make_drift_forecaster("median_all"),
    "Drift: trailing-12-month mean YoY": make_drift_forecaster("mean_trailing12"),
}

validation_tables = {}
for name, forecast_fn in candidate_forecasters.items():
    validation_tables[name] = evaluate(forecast_fn, y, VALIDATION_ORIGINS)

# Choose the drift estimator on validation only, using pooled MAE / pooled seasonal-naive MAE.
drift_names = [name for name in candidate_forecasters if name != "Seasonal Naive"]
validation_scores = {
    name: validation_tables[name]["MAE"].sum() / validation_tables[name]["Benchmark_SN_MAE"].sum()
    for name in drift_names
}
best_drift_name = min(validation_scores, key=validation_scores.get)
print("Validation pooled RelMAE for drift candidates:")
print(pd.Series(validation_scores, name="Pooled RelMAE").sort_values().round(3))
print(f"Selected validation drift baseline: {best_drift_name}")

# After the selection rule is fixed, show all six folds for the seasonal baseline and selected drift.
selected_forecasters = {
    "Seasonal Naive": seasonal_naive_forecast,
    best_drift_name: candidate_forecasters[best_drift_name],
}
all_fold_tables = {
    name: evaluate(forecast_fn, y, ALL_ORIGINS)
    for name, forecast_fn in selected_forecasters.items()
}
results_df = all_fold_tables["Seasonal Naive"][[
    "Origin", "Train_End", "Scored_Months", "MAE", "RMSE", "Bias",
    "December_Abs_Pct_Error"
]].rename(columns={
    "MAE": "SN_MAE", "RMSE": "SN_RMSE", "Bias": "SN_Bias",
    "December_Abs_Pct_Error": "SN_Dec_Abs_Pct_Error",
})
drift_results = all_fold_tables[best_drift_name]
results_df["Selected_Drift_MAE"] = drift_results["MAE"].to_numpy()
results_df["Selected_Drift_RelMAE"] = drift_results["RelMAE_vs_SN"].to_numpy()
results_df["Selected_Drift_Dec_Abs_Pct_Error"] = drift_results["December_Abs_Pct_Error"].to_numpy()
display(results_df.round(3))

# Pre-declared pooled benchmark and fold-win rule; six folds remain a small, noisy sample.
pooled_relmae = drift_results["MAE"].sum() / results_df["SN_MAE"].sum()
fold_wins = int((drift_results["RelMAE_vs_SN"] < 1).sum())
print(f"Pooled RelMAE for selected drift: {pooled_relmae:.3f}")
print(f"Folds beating seasonal naive: {fold_wins}/6")
print("Passes pre-declared bar:", pooled_relmae <= 0.90 and fold_wins >= 4)


Validation pooled RelMAE for drift candidates:
Drift: trailing-12-month mean YoY    0.948
Drift: masked mean YoY               1.086
Drift: masked median YoY             1.173
Name: Pooled RelMAE, dtype: float64
Selected validation drift baseline: Drift: trailing-12-month mean YoY


,Origin,Train_End,Scored_Months,SN_MAE,SN_RMSE,SN_Bias,SN_Dec_Abs_Pct_Error,Selected_Drift_MAE,Selected_Drift_RelMAE,Selected_Drift_Dec_Abs_Pct_Error
0,2016-07,2016-07,12,434.667,564.338,-217.500,3.557,387.981,0.893,2.279
1,2017-07,2017-07,12,590.667,718.648,-261.167,2.606,564.731,0.956,3.499
2,2018-07,2018-07,12,385.167,480.673,-157.833,0.351,425.031,1.103,0.995
3,2019-07,2019-07,7,601.571,709.364,-483.571,3.188,529.464,0.880,2.441
4,2022-07,2022-07,12,526.083,635.634,-142.250,1.165,809.265,1.538,3.531
5,2023-07,2023-07,12,668.000,807.580,-578.333,3.525,549.075,0.822,2.734


Pooled RelMAE for selected drift: 1.019
Folds beating seasonal naive: 4/6
Passes pre-declared bar: False


## Findings & Interpretation

- **Seasonal-naïve baseline:** use the printed fold table for MAE, RMSE, and bias. Negative bias means forecasts are below actual sales; do not claim every fold under-forecasts unless the table confirms it.
- **Drift:** compare the three masked estimators on validation only. The validation winner is carried forward as the selected drift baseline; do not choose it using test-fold results.
- **COVID base effects:** growth observations are excluded if either the current month or its 12-month comparison month falls inside the intervention mask. This prevents the April 2021 rebound from contaminating a drift estimate.
- **December:** interpret `SN_Dec_Abs_Pct_Error` as a percentage, alongside the typical monthly error. One December per fold does not establish a weekend-placement explanation.
- **Uncertainty:** six folds are a small sample. Report pooled RelMAE and the number of folds beaten, not a claim of stable performance across all economic regimes.


## Sanity checks — fold contract and mask behavior

These checks test the function's contract directly rather than rebuilding its training data inside the test. The July 2019 fold forecasts August 2019–July 2020. The scoring mask removes March–July 2020 from that horizon, leaving **7 scored months** (August 2019–February 2020). There are no post-mask months in this fold.


In [3]:
def test_forecast_window_is_aug_to_jul():
    fold = get_forecast_folds(["2016-07-31"])[0]
    assert fold["forecast_months"][0] == pd.Timestamp("2016-08-01")
    assert fold["forecast_months"][-1] == pd.Timestamp("2017-07-01")


def test_folds_stay_inside_data_and_after_training():
    for fold in get_forecast_folds(VALIDATION_ORIGINS + TEST_ORIGINS):
        assert fold["forecast_months"].min() > fold["train_end"]
        assert fold["forecast_months"].max() <= pd.Timestamp("2026-07-01")


def test_scoring_mask_accepts_numpy_predictions():
    dates = pd.date_range("2020-02-01", periods=4, freq="MS")
    actual = pd.Series([1., 2., 3., 4.], index=dates)
    scored_y, scored_pred = apply_scoring_mask(actual, np.array([1., 2., 3., 4.]))
    assert list(scored_y.index) == [pd.Timestamp("2020-02-01")]
    assert scored_y.index.equals(scored_pred.index)


test_forecast_window_is_aug_to_jul()
test_folds_stay_inside_data_and_after_training()
test_scoring_mask_accepts_numpy_predictions()
print("All fold-contract and scoring-mask checks passed.")


All fold-contract and scoring-mask checks passed.
